# B. Descriptive Analytics

## 1- Returns

In [66]:
import pandas as pd

df = pd.read_csv("final_output/psx_final_dataset.csv")

# Ensure correct sorting
df = df.sort_values(["Ticker", "Date"])


In [67]:
#Daily Return
df["Prev_Close"] = df.groupby("Ticker")["Close"].shift(1)
df["Daily_Return"] = (df["Close"] - df["Prev_Close"]) / df["Prev_Close"]

df["Daily_Return"] = df["Daily_Return"].fillna(0)


In [68]:
#WEEKLY RETURN
df["Date"] = pd.to_datetime(df["Date"])
df_weekly = df.set_index("Date").groupby("Ticker")["Close"].resample("W-FRI").last()

df_weekly = df_weekly.to_frame()

df_weekly["Prev_Week_Close"] = df_weekly.groupby("Ticker")["Close"].shift(1)
df_weekly["Weekly_Return"] = (df_weekly["Close"] - df_weekly["Prev_Week_Close"]) / df_weekly["Prev_Week_Close"]
df_weekly["Weekly_Return"] = df_weekly["Weekly_Return"].fillna(0)


In [69]:
#Monthly return
df_monthly = df.set_index("Date").groupby("Ticker")["Close"].resample("ME").last()
df_monthly = df_monthly.to_frame()

df_monthly["Prev_Month_Close"] = df_monthly.groupby("Ticker")["Close"].shift(1)
df_monthly["Monthly_Return"] = (df_monthly["Close"] - df_monthly["Prev_Month_Close"]) / df_monthly["Prev_Month_Close"]
df_monthly["Monthly_Return"] = df_monthly["Monthly_Return"].fillna(0)


In [70]:
#Annual Return
df_yearly = df.set_index("Date").groupby("Ticker")["Close"].resample("YE").last()
df_yearly = df_yearly.to_frame()

df_yearly["Prev_Year_Close"] = df_yearly.groupby("Ticker")["Close"].shift(1)
df_yearly["Annual_Return"] = (df_yearly["Close"] - df_yearly["Prev_Year_Close"]) / df_yearly["Prev_Year_Close"]
df_yearly["Annual_Return"] = df_yearly["Annual_Return"].fillna(0)


In [71]:
#Sample results
print(df[["Date", "Ticker", "Close", "Daily_Return"]].head())


        Date Ticker  Close  Daily_Return
0 2020-01-01   DGKC  72.49      0.000000
1 2020-01-02   DGKC  75.54      0.042075
2 2020-01-03   DGKC  74.80     -0.009796
3 2020-01-06   DGKC  71.06     -0.050000
4 2020-01-07   DGKC  72.49      0.020124


In [72]:
print(df_weekly.head())


                   Close  Prev_Week_Close  Weekly_Return
Ticker Date                                             
DGKC   2020-01-03  74.80              NaN       0.000000
       2020-01-10  73.88            74.80      -0.012299
       2020-01-17  75.50            73.88       0.021927
       2020-01-24  74.75            75.50      -0.009934
       2020-01-31  71.94            74.75      -0.037592


In [73]:
print(df_monthly.head())


                   Close  Prev_Month_Close  Monthly_Return
Ticker Date                                               
DGKC   2020-01-31  71.94               NaN        0.000000
       2020-02-29  66.68             71.94       -0.073116
       2020-03-31  57.09             66.68       -0.143821
       2020-04-30  83.49             57.09        0.462428
       2020-05-31  79.56             83.49       -0.047072


In [74]:
print(df_yearly.head())


                    Close  Prev_Year_Close  Annual_Return
Ticker Date                                              
DGKC   2020-12-31  110.41              NaN       0.000000
       2021-12-31   80.91           110.41      -0.267186
       2022-12-31   51.22            80.91      -0.366951
       2023-12-31   76.77            51.22       0.498829
       2024-12-31  104.07            76.77       0.355608


In [75]:
os.makedirs("final_output/b_discriptive_analytics",exist_ok=True)
df.to_csv("final_output/b_discriptive_analytics/psx_daily_returns.csv", index=False)
df_weekly.to_csv("final_output/b_discriptive_analytics/psx_weekly_returns.csv")
df_monthly.to_csv("final_output/b_discriptive_analytics/psx_monthly_returns.csv")
df_yearly.to_csv("final_output/b_discriptive_analytics/psx_annual_returns.csv")


## 2-volatility per script

In [76]:
# 1-Daily Volatility (standard deviation of daily returns)
vol_daily = df.groupby("Ticker")["Daily_Return"].std().reset_index()
vol_daily = vol_daily.rename(columns={"Daily_Return": "Daily_Volatility"})

#2. Annualized Volatility (used in finance & risk analysis)
import numpy as np

vol_daily["Annualized_Volatility"] = vol_daily["Daily_Volatility"] * np.sqrt(252)


In [77]:
print(vol_daily)


  Ticker  Daily_Volatility  Annualized_Volatility
0   DGKC          0.025342               0.402295
1    HBL          0.019786               0.314096
2   LUCK          0.107317               1.703609
3    MCB          0.015716               0.249478
4   OGDC          0.021954               0.348511


## 3- top gainers/losers for different time periods

### Top Daily Gainers & Losers

In [78]:
# Top 5 Gainers Today
top_daily_gainers = df.sort_values("Daily_Return", ascending=False).head(5)
print("📈 Top Daily Gainers:")
print(top_daily_gainers[["Date", "Ticker", "Close", "Daily_Return"]])


📈 Top Daily Gainers:
           Date Ticker    Close  Daily_Return
4385 2025-03-03   LUCK  1400.89      3.959780
2458 2023-07-31    HBL    77.10      0.145276
7033 2023-07-31   OGDC    81.75      0.135101
7482 2025-05-12   OGDC   196.36      0.100056
1465 2025-09-05   DGKC   235.48      0.100014


In [79]:
# Top 5 Losers Today
top_daily_losers = df.sort_values("Daily_Return").head(5)
print("\n📉 Top Daily Losers:")
print(top_daily_losers[["Date", "Ticker", "Close", "Daily_Return"]])



📉 Top Daily Losers:
           Date Ticker   Close  Daily_Return
4382 2025-02-26   LUCK  289.54     -0.799499
4411 2025-04-10   LUCK  317.68     -0.794767
1580 2020-03-18    HBL   69.38     -0.144407
7480 2025-05-08   OGDC  170.61     -0.087647
1380 2025-05-08   DGKC  118.58     -0.085736


### WEEKLY Gainers/Losers

In [80]:
df["Date"] = pd.to_datetime(df["Date"])

weekly = df.set_index("Date").groupby("Ticker")["Close"].resample("W-FRI").last()
weekly = weekly.to_frame()

weekly["Prev_Week_Close"] = weekly.groupby("Ticker")["Close"].shift(1)
weekly["Weekly_Return"] = (weekly["Close"] - weekly["Prev_Week_Close"]) / weekly["Prev_Week_Close"]
weekly["Weekly_Return"] = weekly["Weekly_Return"].fillna(0)
 

In [81]:
#Top 5 Weekly Gainers
top_weekly_gainers = weekly.sort_values("Weekly_Return", ascending=False).head(5)
print("\n📈 Top Weekly Gainers:")
print(top_weekly_gainers)



📈 Top Weekly Gainers:
                     Close  Prev_Week_Close  Weekly_Return
Ticker Date                                               
LUCK   2025-03-07  1436.81           282.45       4.086953
DGKC   2020-04-03    65.35            51.54       0.267947
HBL    2023-08-04    84.29            67.32       0.252080
       2024-11-29   165.23           132.09       0.250890
OGDC   2023-08-04    89.31            72.02       0.240072


In [82]:
#Top 5 Weekly Losers
top_weekly_losers = weekly.sort_values("Weekly_Return").head(5)
print("\n📉 Top Weekly Losers:")
print(top_weekly_losers)



📉 Top Weekly Losers:
                    Close  Prev_Week_Close  Weekly_Return
Ticker Date                                              
LUCK   2025-02-28  282.45          1410.60      -0.799766
       2025-04-11  317.90          1488.83      -0.786477
OGDC   2024-02-16   93.56           125.76      -0.256043
LUCK   2020-03-20  365.88           489.90      -0.253154
DGKC   2020-03-20   60.77            78.40      -0.224872


### MONTHLY Gainers/Losers

In [83]:
monthly = df.set_index("Date").groupby("Ticker")["Close"].resample("ME").last()
monthly = monthly.to_frame()

monthly["Prev_Month_Close"] = monthly.groupby("Ticker")["Close"].shift(1)
monthly["Monthly_Return"] = (monthly["Close"] - monthly["Prev_Month_Close"]) / monthly["Prev_Month_Close"]
monthly["Monthly_Return"] = monthly["Monthly_Return"].fillna(0)


In [84]:
# Top 5 Monthly Gainers
print("\n📈 Top Monthly Gainers:")
print(monthly.sort_values("Monthly_Return", ascending=False).head(5))



📈 Top Monthly Gainers:
                     Close  Prev_Month_Close  Monthly_Return
Ticker Date                                                 
LUCK   2025-03-31  1484.06            282.45        4.254240
DGKC   2020-04-30    83.49             57.09        0.462428
OGDC   2020-04-30    67.64             48.55        0.393203
HBL    2024-11-30   165.23            119.36        0.384300
DGKC   2023-11-30    72.60             53.23        0.363893


In [85]:
# Top 5 monthly loosers
print("\n📉 Top Monthly Losers:")
print(monthly.sort_values("Monthly_Return").head(5))



📉 Top Monthly Losers:
                    Close  Prev_Month_Close  Monthly_Return
Ticker Date                                                
LUCK   2025-04-30  324.55           1484.06       -0.781309
       2025-02-28  282.45           1170.95       -0.758786
OGDC   2020-03-31   48.55             75.99       -0.361100
HBL    2020-03-31   65.69            100.43       -0.345913
LUCK   2020-03-31  363.92            480.14       -0.242054


### YEARLY Gainers/Losers

In [86]:
yearly = df.set_index("Date").groupby("Ticker")["Close"].resample("YE").last()
yearly = yearly.to_frame()

yearly["Prev_Year_Close"] = yearly.groupby("Ticker")["Close"].shift(1)
yearly["Annual_Return"] = (yearly["Close"] - yearly["Prev_Year_Close"]) / yearly["Prev_Year_Close"]
yearly["Annual_Return"] = yearly["Annual_Return"].fillna(0)


In [87]:
#top 5 gainers
print("\n📈 Top Annual Gainers:")
print(yearly.sort_values("Annual_Return", ascending=False).head(5))


📈 Top Annual Gainers:
                    Close  Prev_Year_Close  Annual_Return
Ticker Date                                              
OGDC   2024-12-31  212.22            97.48       1.177062
DGKC   2025-12-31  221.88           104.07       1.132027
MCB    2024-12-31  251.21           130.65       0.922771
HBL    2025-12-31  305.96           159.26       0.921135
       2023-12-31   91.73            48.56       0.889003


In [88]:
# Top 5 yearly loosers
print("\n📉 Top Annual Losers:")
print(yearly.sort_values("Annual_Return").head(5))



📉 Top Annual Losers:
                    Close  Prev_Year_Close  Annual_Return
Ticker Date                                              
LUCK   2025-12-31  451.58          1091.27      -0.586189
HBL    2022-12-31   48.56            80.35      -0.395644
DGKC   2022-12-31   51.22            80.91      -0.366951
LUCK   2022-12-31  438.41           666.86      -0.342576
DGKC   2021-12-31   80.91           110.41      -0.267186


## 4- sector-wise performance

In [89]:
import pandas as pd

df = pd.read_csv("final_output/psx_final_dataset.csv")

df = df.sort_values(["Ticker", "Date"])
df["Prev_Close"] = df.groupby("Ticker")["Close"].shift(1)
df["Daily_Return"] = (df["Close"] - df["Prev_Close"]) / df["Prev_Close"]
df["Daily_Return"] = df["Daily_Return"].fillna(0)


In [90]:
# Average Daily Return per Sector
sector_avg = df.groupby("Sector")["Daily_Return"].mean().reset_index()
sector_avg = sector_avg.sort_values("Daily_Return", ascending=False)

print("📈 Average Daily Return per Sector:")
print(sector_avg)


📈 Average Daily Return per Sector:
      Sector  Daily_Return
1     Cement      0.001939
0    Banking      0.000971
2  Oil & Gas      0.000934


In [91]:
#Sector Volatility
sector_vol = df.groupby("Sector")["Daily_Return"].std().reset_index()
sector_vol = sector_vol.rename(columns={"Daily_Return": "Sector_Volatility"})

print("\n📉 Sector Volatility:")
print(sector_vol)



📉 Sector Volatility:
      Sector  Sector_Volatility
0    Banking           0.017864
1     Cement           0.077964
2  Oil & Gas           0.021954


In [92]:
# Combine Return + Volatility in one table
sector_perf = sector_avg.merge(sector_vol, on="Sector")
print("\n📊 Sector Performance Summary:")
print(sector_perf)



📊 Sector Performance Summary:
      Sector  Daily_Return  Sector_Volatility
0     Cement      0.001939           0.077964
1    Banking      0.000971           0.017864
2  Oil & Gas      0.000934           0.021954


Identify Top & Worst Sectors

In [93]:
# Best performing sector:
best_sector = sector_perf.sort_values("Daily_Return", ascending=False).head(1)
print("\n🏆 Best Sector:")
print(best_sector)



🏆 Best Sector:
   Sector  Daily_Return  Sector_Volatility
0  Cement      0.001939           0.077964


In [94]:
#Worst performing sector:
worst_sector = sector_perf.sort_values("Daily_Return").head(1)
print("\n💀 Worst Sector:")
print(worst_sector)



💀 Worst Sector:
      Sector  Daily_Return  Sector_Volatility
2  Oil & Gas      0.000934           0.021954


In [95]:
#Sector Total Gains/Losses (Important metric)
sector_close = df.groupby(["Sector", "Date"])["Close"].mean().reset_index()

sector_close["Prev"] = sector_close.groupby("Sector")["Close"].shift(1)
sector_close["Sector_Return"] = (sector_close["Close"] - sector_close["Prev"]) / sector_close["Prev"]
sector_close["Sector_Return"] = sector_close["Sector_Return"].fillna(0)

print("\n📘 Sector Daily Return Trend:")
print(sector_close.head())



📘 Sector Daily Return Trend:
    Sector        Date   Close    Prev  Sector_Return
0  Banking  2020-01-01  95.735     NaN       0.000000
1  Banking  2020-01-02  98.375  95.735       0.027576
2  Banking  2020-01-03  97.425  98.375      -0.009657
3  Banking  2020-01-06  95.965  97.425      -0.014986
4  Banking  2020-01-07  98.280  95.965       0.024123


In [96]:
sector_perf.to_csv("final_output/b_discriptive_analytics/sector_performance.csv", index=False)


## 5- Summarize OHLC data

In [97]:
import pandas as pd

df = pd.read_csv("final_output/psx_final_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])


In [98]:
# — Basic OHLC Summary (per Ticker)
ohlc_summary = df.groupby("Ticker").agg({
    "Open": ["min", "max", "mean"],
    "High": ["min", "max", "mean"],
    "Low": ["min", "max", "mean"],
    "Close": ["min", "max", "mean"],
})


In [99]:
print("📊 OHLC Summary (Min/Max/Mean):")
print(ohlc_summary)


📊 OHLC Summary (Min/Max/Mean):
          Open                         High                          Low  \
           min      max        mean     min      max        mean     min   
Ticker                                                                     
DGKC     38.78   268.79   90.658708   39.66   273.51   92.184361   38.68   
HBL      45.98   312.97  100.228197   47.24   318.98  101.592289   45.15   
LUCK    286.56  1566.76  646.106518  286.56  1608.21  654.364695  272.13   
MCB      57.10   374.61  131.634407   60.38   390.22  133.186564   56.97   
OGDC     47.93   276.43  102.554000   47.93   280.50  103.919770   46.20   

                              Close                       
            max        mean     min      max        mean  
Ticker                                                    
DGKC     259.83   89.131580   39.12   269.19   90.482531  
HBL      309.65   98.781285   45.66   311.90  100.089495  
LUCK    1528.09  637.291567  282.45  1560.96  645.098544  
MCB   

#### 2-Week High & Low (important)
A 52-week high = highest close in past 1 year

A 52-week low = lowest close in past 1 year

In [100]:
df["Year"] = df["Date"].dt.year

high52 = df.groupby("Ticker")["Close"].max().reset_index().rename(columns={"Close": "52_Week_High"})
low52 = df.groupby("Ticker")["Close"].min().reset_index().rename(columns={"Close": "52_Week_Low"})


#### Overall Return (first close → last close)

In [101]:
first_close = df.groupby("Ticker")["Close"].first()
last_close = df.groupby("Ticker")["Close"].last()

overall_return = ((last_close - first_close) / first_close).reset_index()
overall_return.columns = ["Ticker", "Total_Return"]


#### Number of trading days per script

In [102]:
days_count = df.groupby("Ticker")["Date"].count().reset_index()
days_count.columns = ["Ticker", "Trading_Days"]


#### Combine all sumaries in one table

In [103]:
summary = overall_return.merge(high52, on="Ticker")
summary = summary.merge(low52, on="Ticker")
summary = summary.merge(days_count, on="Ticker")

print("📘 Final OHLC Summary Table:")
print(summary)


📘 Final OHLC Summary Table:
  Ticker  Total_Return  52_Week_High  52_Week_Low  Trading_Days
0   DGKC      2.060836        269.19        39.12          1525
1    HBL      1.984976        311.90        45.66          1525
2   LUCK      0.052659       1560.96       282.45          1525
3    MCB      2.990896        373.49        58.15          1525
4   OGDC      1.887829        273.59        47.31          1525


In [104]:
summary.to_csv("final_output/ohlc_summary.csv", index=False)
print("✔ OHLC summary saved to final_output/b_discriptive_analytics/ohlc_summary.csv")


✔ OHLC summary saved to final_output/b_discriptive_analytics/ohlc_summary.csv


## 6- rolling mean and rolling std metrics

In [105]:
import pandas as pd

df = pd.read_csv("final_output/psx_final_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"])


In [106]:
# STEP 2 — Rolling Mean and Std (7-Day Window)
df["RollingMean_7"] = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(window=7).mean())
df["RollingStd_7"]  = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(window=7).std())


In [107]:
#Rolling Mean and Std (14-Day Window)
df["RollingMean_14"] = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(window=14).mean())
df["RollingStd_14"]  = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(window=14).std())


In [108]:
# Rolling Mean and Std (30-Day Window)
df["RollingMean_30"] = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(window=30).mean())
df["RollingStd_30"]  = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(window=30).std())


In [109]:
df[["Date", "Ticker", "Close", "RollingMean_7", "RollingStd_7"]].head(20)


,Date,Ticker,Close,RollingMean_7,RollingStd_7
0,2020-01-01,DGKC,72.49,NaN,NaN
1,2020-01-02,DGKC,75.54,NaN,NaN
2,2020-01-03,DGKC,74.80,NaN,NaN
3,2020-01-06,DGKC,71.06,NaN,NaN
4,2020-01-07,DGKC,72.49,NaN,NaN
5,2020-01-08,DGKC,69.44,NaN,NaN
6,2020-01-09,DGKC,72.90,72.674286,2.078315
7,2020-01-10,DGKC,73.88,72.872857,2.123682
8,2020-01-13,DGKC,73.90,72.638571,1.853703
9,2020-01-14,DGKC,72.70,72.338571,1.597878


In [110]:
df.to_csv("final_output/psx_with_rolling_metrics.csv", index=False)
print("✔ Rolling metrics saved in final_output/b_discriptive_analytics/psx_with_rolling_metrics.csv")


✔ Rolling metrics saved in final_output/b_discriptive_analytics/psx_with_rolling_metrics.csv
